In [ ]:
import numpy as np
import json
import random as random
import sys, os, importlib, math
import matplotlib.pyplot as plt
# import cv2
import torch
from tqdm.auto import tqdm

from matplotlib import rc
rc('text',usetex=True)
rc('text.latex', preamble='\\usepackage{color}')

import shap_bpt as shap_bpt
print(shap_bpt.__version__)

# import ultralytics
from ultralytics import YOLO


In [ ]:
import os
from pathlib import Path
import yaml

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'shap_bpt').is_dir():
            return path
    raise FileNotFoundError('Could not find the project root from the current working directory.')

original_working_dir = Path.cwd()
project_root = find_project_root(original_working_dir)
os.chdir(project_root)
print(f'Project root: {project_root}')

try:
    with open(project_root / "examples/configs/MSCOCO_mac.yaml", "r") as f:
        config = yaml.safe_load(f)
finally:
    os.chdir(original_working_dir)
    print(f'Restored working directory: {original_working_dir}')

dataset_root = config["data"]["dataset_root"]


In [ ]:
import sys

scripts_dir = project_root / "examples/scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import utils_xai as utx
import utils_sam as uts

import importlib
importlib.reload(utx) 
importlib.reload(uts)

In [ ]:
import model_setup as ms

In [ ]:
# !mamba install -y ultralytics

In [ ]:
device = torch.device("cpu")

In [ ]:
path_partition = 'partitions_new_120'


In [ ]:
## Get available precomputed SAM partitions
# fetch available unique image_ids from the partitions directory
image_ids = os.listdir(f"/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/{path_partition}")
image_ids = [f.split('.')[0].split('_')[0] for f in image_ids if '_refined.npy' in f]
print(f'computed image_ids: {len(image_ids)}')

already_computed = ['000000049091',
 '000000000632',
 '000000186929',
 '000000002299',
 '000000225757',
 '000000171382']

image_ids = [img_id for img_id in image_ids if img_id not in already_computed]
print(f'filtered image_ids: {len(image_ids)}')
image_ids

In [ ]:
image_dir = config["data"]["image_dir"] # Update for your image directory
# annotation_file =  config["data"]["annotation_file"] # Update for your annotation file
# coco = COCO(annotation_file)

# categories = coco.loadCats(coco.getCatIds())
# coco_categories = {cat['id']: cat['name'] for cat in categories}



In [ ]:
# image_id = '000000171382'  # Example image ID
image_id = '000000171382'  # Example image ID
# image_id = image_ids[2]  # Example image ID
image_path = os.path.join(image_dir, f'{image_id}.jpg')
image_path

In [ ]:
input_fname = '../imgs/000000011122.jpg'

input_image = ms.load_rgb_image_from_file(image_path)
print('input_image.shape:', input_image.shape, input_image.dtype)
plt.imshow(input_image) ; plt.show()


In [ ]:
class YoloMaskedImageModel(ms.MaskedImageModel):
    def __init__(self, device):
        super().__init__(device, None)
        self.yolo11s_model = YOLO(f'{original_working_dir}/checkpoints/yolo11s.pt')

    def preprocess(self, image):
        return torch.tensor(image).permute(2,0,1).to(self.device)

    def predict(self, x):
        pred = []
        for img in x:
            if isinstance(img, torch.Tensor):
                img = np.moveaxis(img.cpu().detach().numpy(), 0, -1)
                img = (img*255.0).astype(np.uint8)
            res = self.yolo11s_model.predict(source=img, verbose=False)[0]
            p = np.zeros(80)
            for cls,prob in zip(res.boxes.cls.cpu().numpy(), res.boxes.conf.cpu().numpy()):
                p[int(cls)] = prob
            pred.append(np.array(p))
        return np.array(pred)

    def class_name(self, i):
        return self.yolo11s_model.names[i]
    
masker_model = YoloMaskedImageModel(device)
print(masker_model.predict(np.expand_dims(input_image, axis=0)))

In [ ]:
nu = ms.make_characteristic_function_for_image(input_image, masker_model, 'g',
                                                num_explained_classes=4) #explained_class_ids=list(range(80))) # 'bgwsn', 'gs'

MAX_EVALS_BUDGET = 100
batch_size=10 // max(1, len(nu.masked_model.replacement_image_set))
heatmaps = {}
explainer = shap_bpt.Explainer(nu, verbose=True)
print(explainer.image_to_explain.shape)

In [ ]:
heatmaps['AA'] = explainer.explain_instance(MAX_EVALS_BUDGET, method='AA', batch_size=batch_size)#, verbose_plot=True)
heatmaps['BPT'] = explainer.explain_instance(MAX_EVALS_BUDGET, method='BPT', batch_size=batch_size)#, verbose_plot=True)

In [ ]:
shap_bpt.plot_shapley_values(explainer, list(heatmaps.values()), names=list(heatmaps.keys()), 
                             alpha=0.8, cmap=shap_bpt.colormap_default, show_nu_values=True)

In [ ]:
np.sum(heatmaps['AA'])

In [ ]:
pred = np.zeros((100, 200))
truth = np.zeros((100, 200), dtype=bool)

pred[20:50, 30:70] = 1.0
pred[70:80, 100:110] = 0.5
pred[10:90, 150:170] = 0.1
truth[25:45, 35:55] = True


pred = pred + np.random.normal(0.0, 0.0000001, size=truth.shape)

fig,axes = plt.subplots(2,1)
axes[0].imshow(pred)
axes[1].imshow(truth)
plt.show()

In [ ]:
iou = ms.calc_IoU_curve(truth.flatten(), pred.flatten())
display(iou)

In [ ]:
plt.plot(iou['X'], iou['Y'])

In [ ]:
def vis_IoU(shapley_values, threshold, ground_truth, verbose=False):
    pred = shapley_values.flatten() >= threshold
    real = ground_truth.flatten()
    # real = real.astype(np.float32)


    image = np.full((len(pred), 3), 1.0, dtype=np.float32)
    if verbose:
        print(np.sum(pred), np.sum(real))
    image[ pred & real, : ]    = (0.0, 0.0, 0.75) # True Positives
    image[ pred & (~real), : ] = (1.0, 0.6, 0.2)  # False Positives
    image[ (~pred) & real, : ] = (1.0, 0.4, 1.0)  # False Negatives

    return image.reshape(list(ground_truth.shape) + [3])

plt.imshow(vis_IoU(pred, iou['max_IoU_heatmap_threshold'], truth))